# QueryLane specialist fine-tune (Colab T4)

Trains LoRA specialists on the validated synthetic dataset and exports Q4_K_M GGUFs.

**Runtime → Change runtime type → T4 GPU.** Then run cells top to bottom.

- Qwen2.5-**1.5B**: plain LoRA fp16 (~25 min)
- Qwen2.5-**3B**: QLoRA 4-bit (~60-90 min)

Outputs land in your Google Drive under `ft-out/`: `ft-1.5b-q4_k_m.gguf`, `ft-3b-q4_k_m.gguf`.
Also works on Kaggle (2x T4, 30h/week) with the Drive cell swapped for kaggle output.

In [ ]:
!pip install -q "transformers>=4.45" "trl>=0.12" "peft>=0.13" datasets accelerate bitsandbytes gguf sentencepiece
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF > /dev/null && cmake --build /content/llama.cpp/build --target llama-quantize -j2 > /dev/null
print('setup done')

In [ ]:
# Upload train_all.jsonl (from your repo's finetune/data/) when prompted
from google.colab import files
uploaded = files.upload()
DATA = list(uploaded.keys())[0]
print('rows:', sum(1 for _ in open(DATA)))

In [ ]:
import json, torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

def train(base, out, use_qlora, batch, accum):
    tok = AutoTokenizer.from_pretrained(base)
    rows = [json.loads(l) for l in open(DATA) if l.strip()]
    ds = Dataset.from_list([
        {'text': tok.apply_chat_template(r['messages'], tokenize=False, add_generation_prompt=False)}
        for r in rows
    ])
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
                      target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
    model_kwargs = {'torch_dtype': torch.float16}
    if use_qlora:
        model_kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(base, **model_kwargs)
    cfg = SFTConfig(output_dir=out, num_train_epochs=3, learning_rate=2e-4,
                    lr_scheduler_type='cosine', warmup_ratio=0.03,
                    per_device_train_batch_size=batch, gradient_accumulation_steps=accum,
                    fp16=True, logging_steps=25, save_strategy='no',
                    max_seq_length=1024, packing=False, report_to=[])
    trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds, peft_config=lora)
    trainer.train()
    trainer.save_model(out + '/final')
    tok.save_pretrained(out + '/final')
    del trainer, model
    torch.cuda.empty_cache()
    print('saved', out + '/final')

In [ ]:
train('Qwen/Qwen2.5-1.5B-Instruct', '/content/ft15', use_qlora=False, batch=4, accum=4)

In [ ]:
train('Qwen/Qwen2.5-3B-Instruct', '/content/ft3', use_qlora=True, batch=2, accum=8)

In [ ]:
# Merge adapters into fp16 bases (CPU RAM), convert to GGUF, quantize Q4_K_M
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

def merge_convert(base, adapter, tag):
    model = AutoModelForCausalLM.from_pretrained(base, torch_dtype=torch.float16, device_map='cpu')
    model = PeftModel.from_pretrained(model, adapter)
    model = model.merge_and_unload()
    merged = f'/content/merged-{tag}'
    model.save_pretrained(merged)
    AutoTokenizer.from_pretrained(base).save_pretrained(merged)
    del model; torch.cuda.empty_cache()
    !python /content/llama.cpp/convert_hf_to_gguf.py {merged} --outfile /content/{tag}-f16.gguf --outtype f16
    !/content/llama.cpp/build/bin/llama-quantize /content/{tag}-f16.gguf /content/ft-{tag}-q4_k_m.gguf Q4_K_M
    !rm -rf {merged} /content/{tag}-f16.gguf
    !ls -lh /content/ft-{tag}-q4_k_m.gguf

merge_convert('Qwen/Qwen2.5-1.5B-Instruct', '/content/ft15/final', '1.5b')
merge_convert('Qwen/Qwen2.5-3B-Instruct', '/content/ft3/final', '3b')

In [ ]:
# Copy the GGUFs to Google Drive (browser download of 2GB files is flaky)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/ft-out
!cp /content/ft-1.5b-q4_k_m.gguf /content/ft-3b-q4_k_m.gguf /content/drive/MyDrive/ft-out/
!ls -lh /content/drive/MyDrive/ft-out/